In [ ]:
import torch
import numpy as np
from matplotlib import pyplot as plt
from torchvision import datasets, transforms
from rich import print as pr

In [ ]:
torch.manual_seed(0)

tfm = transforms.ToTensor()

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=len(train_ds), shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=len(test_ds),  shuffle=False)

train_images, train_labels = next(iter(train_loader))
test_images,  test_labels  = next(iter(test_loader))

In [ ]:
X = train_images.view(len(train_images), -1).float()   
mean = X.mean(dim=0, keepdim=True)
Xc = X - mean

U, S, Vt = torch.linalg.svd(Xc, full_matrices=False)   

# Statistically Independent Princinple Components

In [ ]:
torch.manual_seed(0)

k = 4
num_samples = 5

z = Xc @ Vt[:k].T  
z_mean = z.mean(0)
z_std = z.std(0)
z_new = torch.randn(num_samples, k) * z_std + z_mean

print(z_std)

X_new = z_new @ Vt[:k] + mean 
X_new = torch.clamp(X_new, 0, 1)

fig, axes = plt.subplots(1,num_samples, figsize=(num_samples*2,2))
axes = [axes] if num_samples == 1 else axes
for i, ax in enumerate(axes):
  ax.imshow(X_new[i].view(28,28))
  ax.axis('off')
plt.tight_layout()
plt.show()

# COVARIANCE METHOD

In [ ]:
torch.manual_seed(0)

k = 4
num_samples = 5

Z = Xc @ Vt[:k].T
z_mean = Z.mean(dim=0)                     
Z_centered = Z - z_mean
cov = (Z_centered.T @ Z_centered) / (Z_centered.shape[0] - 1)   
dist = torch.distributions.MultivariateNormal(z_mean, covariance_matrix=cov)
z_new = dist.sample((num_samples,))          

X_new = z_new @ Vt[:k] + mean                  
X_new = torch.clamp(X_new, 0, 1)

fig, axes = plt.subplots(1, num_samples, figsize=(num_samples * 2, 2))
axes = [axes] if num_samples == 1 else axes
for i, ax in enumerate(axes):
    ax.imshow(X_new[i].view(28, 28).detach().numpy())
    ax.axis('off')
plt.tight_layout()
plt.show()


# Principle components probing

In [ ]:
torch.manual_seed(1)

k = 2
num_samples = 10


N = 15
M = 15
anchor = 7
vals1 = torch.linspace(-anchor, anchor, N)   
vals2 = torch.linspace(-anchor, anchor, M)   
Z1, Z2 = torch.meshgrid(vals1, vals2, indexing="xy")
Z = torch.stack([Z1, Z2], dim=2)

Z_flat = Z.reshape(-1,2)
X_new = Z_flat @ Vt[:k] + mean
X_new = torch.clamp(X_new, 0, 1)
X_new = X_new.reshape(M, N, -1)

fig, axes = plt.subplots(M,N,figsize=(N,M))
for i in range(M):
    for j in range(N):
        axes[i, j].imshow(X_new[i,j].view(28,28))
        axes[i, j].axis('off')
        
        z1, z2 = Z[i, j].tolist()
        axes[i,j].set_title(f"{z1:.1f},    {z2:.1f}", fontsize=6, pad=2)
plt.show()





# ROUND 2

In [ ]:
import torch
from matplotlib import pyplot as plt
from torchvision import datasets
from rich import print as prin

# NOW I GOT TO LOAD THE DATA

torch.manual_seed(0)

traindata = torch.MNIST.traindata(mnist)
testdata = torch.MNIST.traindata(mnist)
train_loader = datasets.dataloader.load_data(traindata, path="./data", batch_size=len(traindata), shuffle=false)
test_loader = datasets.dataloader.load_data(testdata, path="./data", batch_size=len(testdata), shuffle=false)
train_data, train_labels = next(iter(train_loader))
Y_data, Y_labels = next(iter(test_loader))

# X_data.shape (7000,28,28)
X = train_data.view(len(traindata), -1)
# X.shape (7000, 28*28)
X_mean = X.mean(0)
Xc = X - X_mean

U, S, Vt = torch.linalg.svd(Xc, full_matrices=False)

k = 2
# This is going to be the statisticalaly independant principle components sampling generative model

Z = Xc @ Vt[:k].T # (70000, 28*28) @ (2, 28*28).T = (7000, 2)
Z_mean = Z.mean(0)
Z_std = Z.std(0)







